In [ ]:
import tensorflow as tf
print(tf.__version__)

2.19.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import tensorflow as tf
import os

DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/New folder'
IMG_WIDTH = 224
IMG_HEIGHT = 224
BATCH_SIZE = 32

def load_dataset(directory):
    return tf.keras.utils.image_dataset_from_directory(
        directory,
        labels='inferred',
        label_mode='categorical',  # Use categorical for one-hot encoding
        image_size=(IMG_HEIGHT, IMG_WIDTH),
        interpolation='nearest',
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=42
    )

train_dataset_raw = load_dataset(os.path.join(DATA_DIR, 'train'))
val_dataset_raw = load_dataset(os.path.join(DATA_DIR, 'val'))
test_dataset_raw = load_dataset(os.path.join(DATA_DIR, 'test'))

# Get class names from the training dataset before prefetching
class_names = train_dataset_raw.class_names
class_indices = dict(zip(class_names, range(len(class_names))))


print("Train dataset loaded:", train_dataset_raw)
print("Validation dataset loaded:", val_dataset_raw)
print("Test dataset loaded:", test_dataset_raw)
print("Class Names:", class_names)
print("Class Indices:", class_indices)

Found 1821 files belonging to 3 classes.
Found 231 files belonging to 3 classes.
Found 225 files belonging to 3 classes.
Train dataset loaded: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.uint8, name=None), TensorSpec(shape=(None, 3), dtype=tf.float32, name=None))>
Validation dataset loaded: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.uint8, name=None), TensorSpec(shape=(None, 3), dtype=tf.float32, name=None))>
Test dataset loaded: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.uint8, name=None), TensorSpec(shape=(None, 3), dtype=tf.float32, name=None))>
Class Names: ['Elephant', 'Jaguar', 'Wild Boar']
Class Indices: {'Elephant': 0, 'Jaguar': 1, 'Wild Boar': 2}


In [ ]:
# Get class names from the training dataset
class_names = train_dataset_raw.class_names
class_indices = dict(zip(class_names, range(len(class_names))))

print("Class Names:", class_names)
print("Class Indices:", class_indices)

Class Names: ['Elephant', 'Jaguar', 'Wild Boar']
Class Indices: {'Elephant': 0, 'Jaguar': 1, 'Wild Boar': 2}


In [ ]:
from tensorflow.keras import layers

AUTOTUNE = tf.data.AUTOTUNE
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_dataset = train_dataset_raw.map(preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_dataset = val_dataset_raw.map(preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_dataset = test_dataset_raw.map(preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import Sequential, layers, optimizers

IMG_SHAPE = (224, 224, 3)
base_model = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=IMG_SHAPE
)
base_model.trainable = False


model = Sequential([
    base_model,
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(3, activation='softmax')
])

In [ ]:
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='best_model_checkpoint.keras',
    monitor='val_loss',
    save_best_only=True,
    save_weights_only=False,
    mode='min',
    verbose=1
)

early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)
reducelr_callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=1, min_lr=1e-6
)

callbacks = [checkpoint_callback,reducelr_callback]

In [ ]:
model.compile(
    optimizer=optimizers.Adam(),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.fit(
    train_dataset,
    epochs=20,
    validation_data=val_dataset,
    callbacks=callbacks
)

Epoch 1/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.9016 - loss: 0.7905
Epoch 1: val_loss improved from inf to 0.20740, saving model to best_model_checkpoint.keras
57/57 ━━━━━━━━━━━━━━━━━━━━ 494s 8s/step - accuracy: 0.9026 - loss: 0.7848 - val_accuracy: 0.9870 - val_loss: 0.2074 - learning_rate: 0.0010
Epoch 2/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step - accuracy: 0.9868 - loss: 0.1676
Epoch 2: val_loss improved from 0.20740 to 0.06105, saving model to best_model_checkpoint.keras
57/57 ━━━━━━━━━━━━━━━━━━━━ 10s 176ms/step - accuracy: 0.9868 - loss: 0.1673 - val_accuracy: 0.9913 - val_loss: 0.0610 - learning_rate: 0.0010
Epoch 3/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step - accuracy: 0.9931 - loss: 0.0430
Epoch 3: val_loss did not improve from 0.06105
57/57 ━━━━━━━━━━━━━━━━━━━━ 10s 179ms/step - accuracy: 0.9931 - loss: 0.0429 - val_accuracy: 0.9870 - val_loss: 0.1088 - learning_rate: 0.0010
Epoch 4/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step - accuracy: 0.9940 - loss: 0.0443


In [ ]:

base_model.trainable = True

fine_tune_at = len(base_model.layers) - 50
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(optimizer=optimizers.Adam(1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history_fine_tune = model.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset,
    callbacks=callbacks
)

Epoch 1/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 322ms/step - accuracy: 0.8882 - loss: 0.9605
Epoch 1: val_loss improved from 0.04951 to 0.04008, saving model to best_model_checkpoint.keras
57/57 ━━━━━━━━━━━━━━━━━━━━ 51s 496ms/step - accuracy: 0.8884 - loss: 0.9579 - val_accuracy: 0.9957 - val_loss: 0.0401 - learning_rate: 1.0000e-05
Epoch 2/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step - accuracy: 0.9716 - loss: 0.1801
Epoch 2: val_loss improved from 0.04008 to 0.03786, saving model to best_model_checkpoint.keras
57/57 ━━━━━━━━━━━━━━━━━━━━ 12s 208ms/step - accuracy: 0.9715 - loss: 0.1811 - val_accuracy: 0.9957 - val_loss: 0.0379 - learning_rate: 1.0000e-05
Epoch 3/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step - accuracy: 0.9712 - loss: 0.1545
Epoch 3: val_loss improved from 0.03786 to 0.03485, saving model to best_model_checkpoint.keras
57/57 ━━━━━━━━━━━━━━━━━━━━ 12s 203ms/step - accuracy: 0.9713 - loss: 0.1540 - val_accuracy: 0.9957 - val_loss: 0.0349 - learning_rate: 1.0000e-05
Epoch 4/10
57/57

In [ ]:
base_model.trainable = True
model.compile(
    optimizer=optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.fit(
    train_dataset,
    epochs=15,
    validation_data=val_dataset,
    callbacks=callbacks
)

Epoch 1/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 300ms/step - accuracy: 0.9939 - loss: 0.0163
Epoch 1: val_loss did not improve from 0.01577
57/57 ━━━━━━━━━━━━━━━━━━━━ 47s 458ms/step - accuracy: 0.9939 - loss: 0.0164 - val_accuracy: 0.9913 - val_loss: 0.0219 - learning_rate: 1.0000e-05
Epoch 2/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step - accuracy: 0.9944 - loss: 0.0145
Epoch 2: val_loss did not improve from 0.01577
57/57 ━━━━━━━━━━━━━━━━━━━━ 14s 174ms/step - accuracy: 0.9945 - loss: 0.0145 - val_accuracy: 0.9913 - val_loss: 0.0310 - learning_rate: 1.0000e-05
Epoch 3/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step - accuracy: 0.9939 - loss: 0.0211
Epoch 3: val_loss did not improve from 0.01577
57/57 ━━━━━━━━━━━━━━━━━━━━ 11s 191ms/step - accuracy: 0.9939 - loss: 0.0211 - val_accuracy: 0.9913 - val_loss: 0.0226 - learning_rate: 5.0000e-06
Epoch 4/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step - accuracy: 0.9934 - loss: 0.0209
Epoch 4: val_loss did not improve from 0.01577
57/57 ━━━━━━━━━━━━━━━━━━━━ 9s 

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,461,259 (24.65 MB)

 Trainable params: 2,019,459 (7.70 MB)

 Non-trainable params: 402,880 (1.54 MB)

 Optimizer params: 4,038,920 (15.41 MB)

In [ ]:
import shutil
import os

source_path = '/content/best_model_checkpoint.keras'

destination_path = '/content/drive/MyDrive/animal_classification_model.keras'

shutil.copyfile(source_path, destination_path)

print(f"Model saved to: {destination_path}")

Model saved to: /content/drive/MyDrive/animal_classification_model.keras
